# EXPERIMENTING

In [ ]:
import pandas as pd
import os
import pytz
import statistics
from datetime import datetime
from datetime import timedelta
import numpy as np
import pthelperfunctions as helper

pd.set_option('display.max_rows', None)

#path_mob = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/mobilization_output"
#path_ptot = "/gpfs/gibbs/project/jain_snigdha/shared/EM_PTOT_project/ot_pt_output"

In [ ]:
d_items_df = helper.load_data('mimic','d_items',folder='icu', type='csv')
d_items_df.head()

In [ ]:
d_items_df['category'].value_counts()

In [ ]:
'''
Picked categories which appear to be relevant from above.
'''
categories_to_get = ['Care Plans','Treatments','Generic Proc Note','3-Significant Events','4-Procedures','Case Management']
filtered_items = d_items_df[d_items_df['category']==categories_to_get[5]]
print(filtered_items.shape[0])
filtered_items

In [ ]:
'''
Check all D-ITEMS manually by iterating the cell above and hand select the ones we want.
'''
items_to_get = [229344,229347,229349,229350, #Impoired mobility, Care Plans, chartevents
                224084,224086, 229319, 229321, 229633, 229634, 229635, 229636, 229637, 229638, 229681, #Activity/HLM/AMPAC, Treatments, chartevents
                225474, #Fall, 3-Significant Events, procedureevents
                225414] #Home without service, Case Management, chartevents
filtered_items = d_items_df[d_items_df['itemid'].isin(items_to_get)]

In [ ]:
#Load block data from our project output.
path = os.path.join(helper.path_out, "block_compiled_5.parquet")
block_final = pd.read_parquet(path)

#Separate procedureevents and chartevents
items_chart_df = d_items_df[(d_items_df['linksto']=='chartevents') & d_items_df['itemid'].isin(items_to_get)]
items_proc_df = d_items_df[(d_items_df['linksto']=='procedureevents') & d_items_df['itemid'].isin(items_to_get)]

In [ ]:
#Load mimic chart events
chart_df = helper.load_data("mimic","chartevents", folder='icu')
chart_df.rename(columns={'subject_id': 'patient_id','hadm_id':'hospitalization_id'}, inplace=True)

#Filter for our cohort and needs
chart_mask = \
        chart_df['hospitalization_id'].isin(block_final['hospitalization_id'].astype(int)) &\
        chart_df['itemid'].isin(items_chart_df['itemid']) &\
        chart_df['value'].notna()

filtered_chart_df = chart_df[chart_mask]

#Merge with D_items Labels
filtered_chart_df = filtered_chart_df.merge(
    items_chart_df[['itemid','label']],
    on='itemid',
    how='left')

In [ ]:
#Merge with cohort data
block_final['hospitalization_id'] = block_final['hospitalization_id'].astype(int)
filtered_chart_df = filtered_chart_df.merge(
    block_final[['encounter_block','hospitalization_id','pt_post48_IMV']],
    on='hospitalization_id',
    how='left')

#Pivot Table
pivot_chart_table = filtered_chart_df.pivot_table(
    values='encounter_block',
    index=['label','itemid'],
    columns='pt_post48_IMV',
    aggfunc='nunique',
    fill_value=0
)
pivot_chart_table = pivot_chart_table.reset_index()

#Print DF
print(pivot_chart_table)

In [ ]:
#Load mimic proc events
proc_df = helper.load_data("mimic","procedureevents", folder='icu', type='csv.gz')
proc_df.rename(columns={'subject_id': 'patient_id','hadm_id':'hospitalization_id'}, inplace=True)

#Filter for our cohort and needs
proc_mask = \
        proc_df['hospitalization_id'].isin(block_final['hospitalization_id'].astype(int)) &\
        proc_df['itemid'].isin(items_proc_df['itemid']) &\
        proc_df['value'].notna()

filtered_proc_df = proc_df[proc_mask]

#Merge with D_items Labels
filtered_proc_df = filtered_proc_df.merge(
    items_proc_df[['itemid','label']],
    on='itemid',
    how='left')

#Merge with cohort data
filtered_proc_df = filtered_proc_df.merge(
    block_final[['encounter_block','hospitalization_id','pt_post48_IMV']],
    on='hospitalization_id',
    how='left')

#Pivot Table
pivot_proc_table = filtered_proc_df.pivot_table(
    values='encounter_block',
    index=['label','itemid'],
    columns='pt_post48_IMV',
    aggfunc='nunique',
    fill_value=0
)
pivot_proc_table = pivot_proc_table.reset_index()

#Print DF
print(pivot_proc_table)

Now look for the CLIF Procedure Table

In [ ]:
import pandas as pd
import numpy as np
import os
import pthelperfunctions as helper

path = os.path.join(helper.path_out, "block_compiled_5.parquet")
df = pd.read_parquet(path)
proc_df = helper.load_clif_table('patient_procedures', hosp_list=df['hospitalization_id'].tolist())

codes_list = pd.read_csv(os.path.join("..","config","therapy_codes.csv"))
codes_list['code'] = codes_list['code'].astype(str)
codes_list.rename(columns={'code':'procedure_code'}, inplace=True)
cpt_df = pd.merge(
    proc_df,
    codes_list,
    on='procedure_code',
    how='inner'
)
cpt_df.head(10)

In [ ]:
proc_df['procedure_code_format'].value_counts()